# Load a model and chat

This standalone tutorial loads one Hugging Face model, sends messages, and releases it. No earlier tutorial needs to be run. RAG is not used yet.

On a new computer, first run the setup command in the README. Select QAChat's `.venv` kernel, then run one cell at a time with Shift+Enter.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"Root directory: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


print(f"Python environment: {sys.executable}")

In [ ]:
import platform

print(f"Computer system: {platform.system()}")
print(f"Computer machine: {platform.machine()}")

## 1. See models already cached

This reads Hugging Face's shared cache. A model downloaded by another Hugging Face application can appear here.

Each printed name is a **model ID**. Copy the model ID you want into `MODEL_ID` in the next section. For example: `MODEL_ID = "lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly"`.

This is a list of cached **model candidates**, not a list of guaranteed chat models. The cache may also contain embedding, image, or incompatible models. Choose a text-generation/chat model whose weights match MLX or PyTorch.

In [ ]:
from qachat.huggingface import available_models

models = available_models()
if models:
    print("Model IDs found in the Hugging Face cache:")
    for number, model in enumerate(models, start=1):
        size_gb = model["size_bytes"] / 1_000_000_000
        print(f"{number}. {model['model_id']} ({size_gb:.2f} GB)")
else:
    print("No models are cached yet.")

## 2. Choose the model and engine

Use `auto`, `mlx`, or `torch`. On Apple Silicon, `auto` chooses MLX.

In [ ]:
BACKEND = "auto"

To use a cached model, copy its exact ID from the list above:

```python
MODEL_ID = "mlx-community/Qwen3-14B-4bit"
```

`THINKING` is passed to compatible chat templates. Temperature `0` gives repeatable greedy output. `MAX_NEW_TOKENS = None` adds no QAChat length cap; generation uses the model's remaining context. Set a positive number for a shorter experiment.

If `ALLOW_DOWNLOAD` is true and files are missing, Hugging Face downloads them to its normal cache during the loading cell.

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B-MLX-4bit"
THINKING = False
TEMPERATURE = 0.0
MAX_NEW_TOKENS = None
ALLOW_DOWNLOAD = True

## 3. Load the selected model

This is the only cell that loads weights. Loading can use several GB of memory. It closes an earlier session before creating a new one.

In [ ]:
if "session" in globals():
    session.close()

In [ ]:
from qachat.chat import ChatSession

session = ChatSession(
    MODEL_ID,
    backend=BACKEND,
    thinking=THINKING,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    system_prompt="Answer clearly and briefly.",
    allow_download=ALLOW_DOWNLOAD,
)

In [ ]:
print(f"Loaded model: {session.model_id}")
print(f"Backend: {session.backend}")
print(f"Device: {session.device}")

## 4. Chat

The second message receives the earlier turn as conversation history. Change the text and rerun either cell to continue.

In [ ]:
answer = session.reply(
    "Hello, I am Iok-ui! Please, greet me in one short sentence."
)

assert answer.strip()

print(answer)

In [ ]:
answer = session.reply(
    "Copy the name from my first message. Answer with the name only."
)

assert "Iok-ui" in answer

print(answer)

## 5. Release the model

Closing releases this session's model memory and clears its messages. Cached model files remain available for later use.

In [ ]:
session.close()

print("Chat closed.")